# 1 · Outcomes  `[EVAL]`

Full-conversation eval outcomes (the held-out measure, never the training reward) — **Level 1 of the drill-down: the global scores** (one number per conversation per questionnaire). The all-metric trajectory grid (THE main figure), a per-metric learning-curve catalog (`trajectories/`), and every endpoint artifact as a **final + best pair** (best = each arm's peak iteration on its own training oracle — the checkpoint you would actually select; GRPO peaks at iter 8 then regresses). Exports → `results/<VIEW>/figures|tables/1_outcomes/`; the presentation copies also land in `0_headline/`.

Drill deeper: per-questionnaire items/subscales → `2_Questionnaire_Detail`, validity + reward-hacking → `3_Validity_and_Hacking`, persona splits → `4_Heterogeneity`, stats tables → `7_Stats`.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

# Secondary knobs — defaults are fine. ks/methods left default so VIEW drives arm selection
# (set ks=[...] only to override the view's K filter). All fields: eda_analysis/config.py.
cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="1_outcomes",             # topic family = this notebook's number · figs=PNG, tables=md+xlsx
    selection="all",                       # "all" | "best" (best iter per arm by own oracle)
    focus_arms=None, focus_metric="Q1Q2",  # default arms/metric for single-metric figures
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())   # arms shown in the trajectory figures
# Best iteration per arm (own training oracle — the checkpoint you'd select); drives the *_best figures.
BEST = eda_analysis.best_iteration_by_arm(S.SCORES)
print("best iteration per arm:", BEST)

## 1 · Outcome trajectories — all metrics  `[EVAL]`
Per-metric mean ± 95% CI across iterations, arms overlaid — the one-glance overview. Global-eval (halo) rubrics beside the orthogonal axes (PCT, MICI ↓); caveat under the grid.

In [ ]:
fig = plots.trajectory_grid(S.SCORES, palette=S.PALETTE, arms=cfg.focus_arms)
eda_analysis.save_fig(fig, "trajectories_all_metrics", caption="Full-conv eval: per-metric mean +/- 95% CI across iterations, arms overlaid (all 9 metrics incl. the orthogonal axes).")
eda_analysis.save_fig(fig, "trajectories_all_metrics", group="0_headline", caption="THE main figure — headline copy; canonical: 1_outcomes/."); plt.show()

## 2 · Per-metric learning curves  `[EVAL]`
One full-size curve per metric → `figures/1_outcomes/trajectories/`. Peaks that precede the final iteration are auto-flagged ("peak → regresses" — GRPO's iter-8 peak). The oracle-noise band draws only on Q1+Q2 (the ~0.10 reproducibility figure was measured on that rubric).

In [ ]:
for m in S.METRICS:
    fig = plots.single_metric_trajectory(S.SCORES, m, palette=S.PALETTE, arms=cfg.focus_arms,
                                         oracle_noise=(S.ORACLE_NOISE if m == "Q1Q2" else None),
                                         mark_peaks=True)
    eda_analysis.save_fig(
        fig, f"trajectory_{m}", group="1_outcomes/trajectories",
        caption=f"{eda_analysis.display_label(m)} across iterations per arm (mean +/- 95% CI, N=96); "
                "dotted vline flags a peak that precedes the final iteration (regression)."
                + (" Grey band = oracle reproducibility (~0.10) around base." if m == "Q1Q2" else ""))
    plt.show()

## 3 · Did it work? — effect vs base, FINAL + BEST  `[EVAL]`
The vs-base effect per arm × metric as a forest plot (all 9 metrics; MI-Inconsistency ↓ rows are valence-inverted — red = moved the wrong way), reported twice: at the matched **final** iteration and at each arm's **best** iteration (own oracle; GRPO's best = iter 8 — the honest model-selection view). Full stats tables in `7_Stats`.

In [ ]:
for target, iters_note in (("final", "matched FINAL iteration"),
                           ("best", "each arm's BEST iteration (own oracle)")):
    MR = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target=target), S.SCORES)
    fig = plots.effect_forest(MR, title=f"Effect on full-conversation eval vs base — {iters_note}")
    eda_analysis.save_fig(fig, f"effect_vs_base_forest_{target}",
                          caption=f"Improvement vs base (Δ + 95% CI) per arm x rubric at the {iters_note}, "
                                  "full-conv eval; dot color = effect-size label, dz annotated.")
    eda_analysis.save_fig(fig, f"effect_vs_base_forest_{target}", group="0_headline",
                          caption=f"Effect vs base at the {iters_note} — headline copy; canonical: 1_outcomes/.")
    plt.show()

## 4 · Endpoint bars — FINAL + BEST models  `[EVAL]`
Per-metric bars over the selected checkpoints only (arm-bases pooled into one descriptive `Base`, dotted base line): once at each arm's **final** iteration, once at its **best** (own-oracle) iteration. The all-iteration story lives in the trajectory grid (§1), so no exhaustive every-model wall here.

In [ ]:
for target, select, note in (("final", eda_analysis.final_per_experiment, "final iteration"),
                             ("best", eda_analysis.best_per_experiment, "best iteration (own oracle)")):
    D = eda_analysis.collapse_base(select(S.SCORES)[0])
    fig = plots.outcomes_by_model(D, palette=figures.arm_palette(sorted(D.arm.unique())),
                                  order=figures.model_order(D),
                                  title=f"Outcome metrics at each arm's {note} — full-conversation eval (dotted = base)")
    eda_analysis.save_fig(fig, f"outcomes_by_model_{target}",
                          caption=f"Each arm at its {note} x metrics; mean +/- 95% CI over 96 personas "
                                  "(arm-bases pooled into Base; dotted line = base).")
    plt.show()

## 5 · Scorecard — FINAL + BEST, global-eval rubrics vs orthogonal axes  `[EVAL]`
Each arm's score per metric at its **final** and **best** iteration (one table, `target` column): the global-eval (halo) rubrics beside `PCT`, `MICI ↓`, and the derived `R:Q`/`%CR`/`%MICO`. **Read:** the halo cluster can rise while technique / patient-outcome lag — the multi-skill story the global-eval rubrics hide; the final-vs-best gap is GRPO's post-peak regression.

In [ ]:
LB = pd.concat([plots.leaderboard_scorecard(S.SCORES, selection="final").assign(target="final"),
                plots.leaderboard_scorecard(S.SCORES, selection="best").assign(target="best")],
               ignore_index=True)
LB = LB[["target"] + [c for c in LB.columns if c != "target"]]
display(LB)
eda_analysis.save_table(LB, "leaderboard_scorecard", caption="Final AND best iteration per arm (target column): global-eval (halo) rubrics beside the orthogonal axes (MICI lower-is-better, flagged ↓).")
eda_analysis.save_table(LB, "leaderboard_scorecard", group="0_headline", caption="Scorecard (final + best) — headline copy; canonical: tables/1_outcomes/.")

## 6 · Artifact index
Refresh the per-view `results/<VIEW>/INDEX.md` (every notebook ends with this — whichever ran last completes the map).

In [ ]:
print("index ->", eda_analysis.build_index())